#The Encoder-Decoder Architecture

As we move toward Transformers, we must first understand the Encoder-Decoder (Seq2Seq) framework. This is the foundation for tasks like Machine Translation (e.g., English to Hindi) or Text Summarization.

#How it Works:

1. **Encoder:** Processes the input sequence (e.g., "How are you") and compresses the information into a single Context Vector.

2. **The Bottleneck:** The context vector is the only information the decoder gets. If the sentence is too long, the vector can't hold all the details (this led to the invention of Attention).

3. **Decoder:** Takes the context vector and generates the output sequence (e.g., "आप कैसे हैं") word by word.

1. Implementation (Conceptual Architecture)

In Keras, this involves defining two distinct models that work together.

In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

# Hyperparameters
latent_dim = 256  # Latent dimensionality of the encoding space

# --- ENCODER ---
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(input_dim=5000, output_dim=128)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

# We discard `encoder_outputs` and only keep the states.
encoder_states = [state_h, state_c]

# --- DECODER ---
decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(input_dim=5000, output_dim=128)
dec_emb = dec_emb_layer(decoder_inputs)

# We set up our decoder to return full output sequences, 
# and to return internal states as well.
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(5000, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# --- FULL MODEL ---
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 128) │    640,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 128) │    640,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    394,240 │ embedding[0][0]   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    394,240 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  1,285,000 │ lstm_1[0][0]      │
│                     │ 5000)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,353,480 (12.79 MB)

 Trainable params: 3,353,480 (12.79 MB)

 Non-trainable params: 0 (0.00 B)